# 📈 Stock Price Prediction using Machine Learning
**Objective:** Use historical stock data to predict the next day's closing price.

**Stock Selected:** Apple Inc. (AAPL)

**Models Used:** Linear Regression & Random Forest

---

## 📦 Step 1: Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline

print('✅ Libraries imported successfully!')

---
## 📂 Step 2: Load Stock Data from Yahoo Finance

In [ ]:
# Fetch Apple stock data for the last 2 years
ticker = 'AAPL'
df = yf.download(ticker, start='2022-01-01', end='2024-12-31')

# Flatten multi-level columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'✅ Data loaded for {ticker}')
print(f'Total records: {len(df)}')
df.head()

---
## 🔍 Step 3: Inspect the Dataset

In [ ]:
# Shape and columns
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Summary statistics
df.describe()

---
## 📊 Step 4: Visualize Historical Closing Price

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.index, df['Close'], color='royalblue', linewidth=1.5)
plt.title(f'{ticker} Closing Price (2022–2024)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## ⚙️ Step 5: Feature Engineering
We use **Open, High, Low, Volume** to predict next day's **Close** price.

In [ ]:
# Create target: next day's closing price
df['Target'] = df['Close'].shift(-1)

# Drop last row (no target available)
df.dropna(inplace=True)

# Features and target
features = ['Open', 'High', 'Low', 'Volume']
X = df[features]
y = df['Target']

print('✅ Features shape:', X.shape)
print('✅ Target shape:', y.shape)
X.head()

---
## ✂️ Step 6: Split Data into Train & Test Sets

In [ ]:
# 80% train, 20% test — no shuffle (time series order matters!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

---
## 🤖 Step 7: Train Models

In [ ]:
# ── Linear Regression ──
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
print('✅ Linear Regression trained!')

# ── Random Forest ──
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
print('✅ Random Forest trained!')

---
## 📏 Step 8: Evaluate Models

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'--- {name} ---')
    print(f'  MAE  : {mae:.4f}')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  R²   : {r2:.4f}')
    print()

evaluate('Linear Regression', y_test, lr_preds)
evaluate('Random Forest',     y_test, rf_preds)

---
## 📉 Step 9: Plot Actual vs Predicted Closing Prices

In [ ]:
test_dates = df.index[-len(y_test):]

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Linear Regression Plot
axes[0].plot(test_dates, y_test.values, label='Actual',    color='royalblue',  linewidth=1.5)
axes[0].plot(test_dates, lr_preds,      label='Predicted', color='orangered',   linewidth=1.5, linestyle='--')
axes[0].set_title('Linear Regression — Actual vs Predicted', fontsize=13)
axes[0].set_ylabel('Price (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Random Forest Plot
axes[1].plot(test_dates, y_test.values, label='Actual',    color='royalblue',  linewidth=1.5)
axes[1].plot(test_dates, rf_preds,      label='Predicted', color='seagreen',    linewidth=1.5, linestyle='--')
axes[1].set_title('Random Forest — Actual vs Predicted', fontsize=13)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Price (USD)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{ticker} — Next Day Closing Price Prediction', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🌲 Step 10: Feature Importance (Random Forest)

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(x=importance.values, y=importance.index, palette='viridis')
plt.title('Feature Importance — Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## ✅ Summary

| Step | What We Did |
|------|-------------|
| Load Data | Fetched Apple stock data using `yfinance` |
| Inspect | Checked shape, missing values, statistics |
| Features | Used Open, High, Low, Volume to predict next Close |
| Models | Trained Linear Regression & Random Forest |
| Evaluate | Compared MAE, RMSE, R² scores |
| Plot | Visualized Actual vs Predicted prices |
| Importance | Identified most impactful features |

> **💡 Tip:** Random Forest generally performs better than Linear Regression for stock data due to its ability to capture non-linear patterns.